# Quantitative Evaluation of LF-7T-CycleGAN

This notebook gathers the metrics for quantitative evaluation of LF-7T-CycleGAN through comparison with baseline models.

### Configuration

Prior to execution, all comparison models must first generate 7T predictions of the same 0.3T test dataset.

For the CycleGAN based methods, please reference the `.npy` files in the `npy` folder of the test's `test_latets` directory. For SRDDL, predictions must be copied to a new directory with two sub directories: `HR` and `HR-PRED`, containing the true 7T scans and predicted 7T scans respectivley. Identical file names for the real 0.3T scan and predicted 7T image must be used between the two sub-directories.

In [86]:
# Change these to point to the ouput folders for each model

LF_7T_CYCLEGAN_RESULTS_T1W = "G:\\LF-7T-CycleGAN\\pytorch-cyclegan-and-pix2pix\\results\\T1w-ensemble-results\\test_latest\\npy"
LF_7T_CYCLEGAN_RESULTS_T2W = "G:\\LF-7T-CycleGAN\\pytorch-cyclegan-and-pix2pix\\results\\T2w-ensemble-results\\test_latest\\npy"

UNMODIFIED_CYCLEGAN_RESULTS_T1W = "G:\\pytorch-CycleGAN-and-pix2pix\\results\\T2w-basic-cycleGAN\\test_latest\\npy"
UNMODIFIED_CYCLEGAN_RESULTS_T2W = "G:\\pytorch-CycleGAN-and-pix2pix\\results\\T1w-basic-cycleGAN\\test_latest\\npy"

SRDDL_RESULTS_T1W = "G:\\LF-7T-CycleGAN\\pytorch-cyclegan-and-pix2pix\\results\\SRDDL-evaluation-dataset\\T1w"
SRDDL_RESULTS_T2W = "G:\\LF-7T-CycleGAN\\pytorch-cyclegan-and-pix2pix\\results\\SRDDL-evaluation-dataset\\T2w"

In [ ]:
# Import required libraries

import glob
import os
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import torch

### Helper functions for normalising `.npy` files and discovering test image paths

In [88]:
# Load the prediction data from each dataset

def normalise_npy_file(data):
    """
    Normalises a numpy file to [-1 to 1]. Used so model poutputs can be compared from different models.

    Args:
        data: raw .npy file data

    Returns:
        data: .npy file data normalised to values between -1 and 1

    """
    
    data = data - data.min()

    data = data / data.max()

    data = data * 2 - 1


    return data


def load_cycleGAN_results(results_dir):
    """
    Provides lists of file paths to the real 7T and predicted 7T images for the CycleGAN-based models

    Args:
        results_dir (string): The path of the npy folder of the cycleGAN model outputs

    Returns:
        tuple: (reals[], preds[]) A tuple containing two matching arrays of real paths and predicted paths

    """

    reals = glob.glob(os.path.join(results_dir, '*real_B*'))

    preds = [real_path.replace('real_B', 'rec_B') for real_path in reals]

    return ( reals, preds )


def load_SRDDL_results(results_dir):
    """
    Provides lists of file paths to the real 7T and predicted 7T images for the SRDDL baseline model

    Args:
        results_dir (string): The path of the npy folder of the SRDDL model outputs

    Returns:
        tuple: (reals[], preds[]) A tuple containing two matching arrays of real paths and predicted paths

    """

    hr_dir = os.path.join(results_dir, 'HR')
    pred_dir = os.path.join(results_dir, 'PRED-HR')

    reals = sorted(glob.glob(os.path.join(hr_dir, '*.npy')))
    preds = [os.path.join(pred_dir, os.path.basename(p)) for p in reals]

    return ( reals, preds )

### Utility funtion to calculate metrics for a given method

In [ ]:
# Initialise the LPIPS model once
lpips_func = lpips.LPIPS(net='alex', verbose=False)


def evaluate_model(preds, reals):
    """
    Prints PSNR, SSIM and LPIPS scores for the given predicted and real paths

    Args:
        preds: list of file paths to the predicted 7T images
        reals: list of file paths to the real 7T images

    """

    # Initialise lists to store scores
    psnr_scores, ssim_scores, lpips_scores = [], [], []

    # Zip real and predicted scan paths and compute scores for each pair
    for real_path, pred_path in zip(reals, preds):

        # Normalise both files to -1 to 1 for fair comparison
        real = normalise_npy_file(np.load(real_path).astype(np.float32))
        pred  = normalise_npy_file(np.load(pred_path).astype(np.float32))

        # Pre-process for LPIPS
        real = np.stack([real] * 3, axis=-1)
        pred  = np.stack([pred]  * 3, axis=-1)
        real_tensor = torch.tensor(real).permute(2, 0, 1).unsqueeze(0) * 2 - 1
        pred_tensor  = torch.tensor(pred).permute(2, 0, 1).unsqueeze(0) * 2 - 1

        # Calculate scores and append to lists
        psnr_scores.append(psnr(real, pred, data_range=1.0))
        ssim_scores.append(ssim(real, pred, data_range=1.0, channel_axis=-1))
        lpips_scores.append(lpips_func(real_tensor, pred_tensor).item())

    # Print cumulative results
    print(f"    N: {len(psnr_scores)}")
    print(f"    PSNR: {np.mean(psnr_scores):.2f}, std dev: {np.std(psnr_scores):.2f}")
    print(f"    SSIM: {np.mean(ssim_scores):.2f}, std dev: {np.std(ssim_scores):.2f}")
    print(f"    LPIPS: {np.mean(lpips_scores):.2f}, std dev: {np.std(lpips_scores):.2f}")

In [ ]:
# Load the paths for all three models in each contrast

# LF-7T-CycleGAN
LF_7T_CYCLEGAN_T1W_REALS, LF_7T_CYCLEGAN_T1W_PREDS = load_cycleGAN_results(LF_7T_CYCLEGAN_RESULTS_T1W)
LF_7T_CYCLEGAN_T2W_REALS, LF_7T_CYCLEGAN_T2W_PREDS = load_cycleGAN_results(LF_7T_CYCLEGAN_RESULTS_T2W) 

# Un-modified CycleGAN
UNMODIFIED_CYCLEGAN_T1W_REALS, UNMODIFIED_CYCLEGAN_T1W_PREDS = load_cycleGAN_results(UNMODIFIED_CYCLEGAN_RESULTS_T1W)
UNMODIFIED_CYCLEGAN_T2W_REALS, UNMODIFIED_CYCLEGAN_T2W_PREDS = load_cycleGAN_results(UNMODIFIED_CYCLEGAN_RESULTS_T2W)

# SRDDL
SRDDL_T1W_REALS, SRDDL_T1W_PREDS = load_SRDDL_results(SRDDL_RESULTS_T1W)
SRDDL_T2W_REALS, SRDDL_T2W_PREDS = load_SRDDL_results(SRDDL_RESULTS_T2W)

## Compute PSNR, SSIM and LPIPS for each model

In [91]:
print(f"\n\n{'='*20}  LF-7T-CycleGAN T1w  {'='*20}")
evaluate_model(LF_7T_CYCLEGAN_T1W_REALS, LF_7T_CYCLEGAN_T1W_PREDS)

print(f"\n\n{'='*20}  LF-7T-CycleGAN T2w  {'='*20}")
evaluate_model(LF_7T_CYCLEGAN_T2W_REALS, LF_7T_CYCLEGAN_T2W_PREDS)

print(f"\n\n{'='*20}  Un-modified CycleGAN T1w {'='*20}")
evaluate_model(UNMODIFIED_CYCLEGAN_T1W_REALS, UNMODIFIED_CYCLEGAN_T1W_PREDS)

print(f"\n\n{'='*20}  Un-modified CycleGAN T2w {'='*20}")
evaluate_model(UNMODIFIED_CYCLEGAN_T2W_REALS, UNMODIFIED_CYCLEGAN_T2W_PREDS)

print(f"\n\n{'='*20}  SRDDL T1w {'='*20}")
evaluate_model(SRDDL_T1W_REALS, SRDDL_T1W_PREDS)

print(f"\n\n{'='*20}  SRDDL T2w {'='*20}")
evaluate_model(SRDDL_T2W_REALS, SRDDL_T2W_PREDS)



====================  LF-7T-CycleGAN T1w  ====================
    N: 370
    PSNR: 21.70, std dev: 0.86
    SSIM: 0.86, std dev: 0.03
    LPIPS: 0.05, std dev: 0.01


====================  LF-7T-CycleGAN T2w  ====================
    N: 370
    PSNR: 21.57, std dev: 1.09
    SSIM: 0.84, std dev: 0.04
    LPIPS: 0.07, std dev: 0.02


====================  Un-modified CycleGAN T1w ====================
    N: 370
    PSNR: 24.18, std dev: 1.08
    SSIM: 0.91, std dev: 0.03
    LPIPS: 0.05, std dev: 0.01


====================  Un-modified CycleGAN T2w ====================
    N: 370
    PSNR: 24.23, std dev: 1.11
    SSIM: 0.90, std dev: 0.02
    LPIPS: 0.04, std dev: 0.01


====================  SRDDL T1w ====================
    N: 370
    PSNR: 14.16, std dev: 1.40
    SSIM: 0.70, std dev: 0.05
    LPIPS: 0.12, std dev: 0.02


====================  SRDDL T2w ====================
    N: 370
    PSNR: 18.59, std dev: 1.35
    SSIM: 0.78, std dev: 0.05
    LPIPS: 0.11, std dev: 0.03
